In [ ]:

"""
button_context_job_scraper.py

Notebook / script friendly — designed to paste into a Jupyter cell or save as a single Python file.

Purpose:
 - For each company careers URL, find every actionable element on the page (anchors, buttons, inputs with type=button/submit, elements with role="button", etc.)
 - For each element, capture a rich context: element text, href, outer HTML snippet, ancestor container text, nearby headings, and attributes.
 - Build candidates from those elements, then filter:
    * Keep candidates that look job-related (contain "apply", "job", etc.) OR match user keywords (e.g., "navigation", "gnc", "engineer")
    * Exclude obvious banners/navigation/marketing blocks even if they contain keywords (heuristics)
 - From surviving candidates, try to extract structured fields (title, location, req id, salary, date) from the context using regex heuristics.
 - Deduplicate using a persistent pickle file so you only get new postings.
 - Compose and send an email with newly found matches.

Notes:
 - This intentionally targets buttons/anchors as the source of truth (the "Apply" / "View" call-to-action).
 - It uses requests + BeautifulSoup for speed and an optional Selenium fallback for JS-rendered pages.
 - It's rule-based and heuristic-driven. A machine learning solution is possible but would require labeled training data and infrastructure; see the bottom of this file for a short discussion.
"""

from typing import List, Dict, Optional, Tuple
import requests, time, re, pickle, os, json
from bs4 import BeautifulSoup, Tag
from urllib.parse import urljoin, urlparse
from dataclasses import dataclass, asdict
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import smtplib

# Optional selenium imports (only used if selenium_fallback=True)
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except Exception:
    SELENIUM_AVAILABLE = False

print(SELENIUM_AVAILABLE)
# -------------------------
# Configuration / Defaults
# -------------------------
DEFAULT_SEEN_PATH = "seen_jobs_buttons.pkl"
USER_AGENT = "Mozilla/5.0 (compatible; JobButtonScraper/1.0)"
REQUEST_TIMEOUT = 12
SELENIUM_WAIT = 6

# Heuristics sets
APPLY_TERMS = {"apply", "apply now", "view job", "view position", "job details", "see details", "apply online"}
JOB_KEYWORDS = {"engineer","engineers","engineering","aerospace","gnc","guidance","navigation","control",
                "propulsion","satellite","orbit","software","systems","analyst","intern","developer",
                "operations","test","manufacturing","rf","payload","mechanical","electrical","mission"}
BANNER_INDICATORS = {"header","footer","nav","menu","banner","hero","carousel","promo","promo-box","subscribe","breadcrumb"}
BAD_LINK_FRAGMENTS = {"about","contact","privacy","terms","product","products","services","blog","news","shop","store","docs"}
MIN_TITLE_WORDS = 2

# regex helpers for extracting fields
REQ_ID_RE = re.compile(r"\b(R|REQ|Req|Requisition|Job\s*ID)\s*[#:-]?\s*([A-Za-z0-9-]+)\b", re.I)
LOCATION_RE = re.compile(r"\b([A-Z][a-zA-Z]+(?:[ \-/][A-Z][a-zA-Z]+)*),\s*([A-Z]{2}|[A-Za-z]{2,})\b")  # "City, ST"
SALARY_RE = re.compile(r"(\$|£|€)\s*\d[\d,\.]*")
DATE_RE = re.compile(r"(posted|date|published|posted on)[:\s-]*([A-Za-z0-9 ,./\-]+)", re.I)

# -------------------------
# Data classes
# -------------------------
@dataclass
class Candidate:
    company: str
    base_url: str
    element_tag: str
    element_text: str
    href: Optional[str]
    outer_html: str
    ancestor_text: str
    nearby_headings: List[str]
    attributes: Dict[str,str]

@dataclass
class JobMatch:
    company: str
    title: str
    link: str
    location: Optional[str] = None
    salary: Optional[str] = None
    req_id: Optional[str] = None
    date_posted: Optional[str] = None
    context_html: Optional[str] = None

# -------------------------
# Utilities
# -------------------------
def load_seen(path: str = DEFAULT_SEEN_PATH) -> set:
    if os.path.exists(path):
        try:
            with open(path,"rb") as f:
                return pickle.load(f)
        except Exception:
            return set()
    return set()

def save_seen(seen:set, path: str = DEFAULT_SEEN_PATH):
    with open(path,"wb") as f:
        pickle.dump(seen, f)

def fetch_html_requests(url: str, timeout:int = REQUEST_TIMEOUT) -> Optional[str]:
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent": USER_AGENT})
        r.raise_for_status()
        return r.text
    except Exception as e:
        # print(f"[fetch_requests] {url} failed: {e}")
        return None

def fetch_html_selenium(url: str, wait:int = SELENIUM_WAIT) -> Optional[str]:
    if not SELENIUM_AVAILABLE:
        print("[fetch_html_selenium] Selenium not available in environment.")
        return None
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    # keep driver downloads minimal
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    try:
        driver.get(url)
        time.sleep(wait)
        html = driver.page_source
        return html
    except Exception as e:
        print(f"[fetch_html_selenium] failed for {url}: {e}")
        return None
    finally:
        try:
            driver.quit()
        except Exception:
            pass

def is_banner_element(tag: Tag) -> bool:
    """
    Heuristic checks if an element is likely a nav/banner rather than a job card:
    - If element or any ancestor has classes/ids containing banner indicators
    - If element is inside <nav>, <header>, <footer>, <aside>
    """
    if not isinstance(tag, Tag):
        return False
    # check tag names up the tree
    for anc in tag.parents:
        name = getattr(anc, "name", "")
        if name in ("nav","header","footer","aside"):
            return True
        cls = " ".join(anc.get("class",[]) if isinstance(anc.get("class"), list) else ([anc.get("class")] if anc.get("class") else []))
        idv = anc.get("id","") or ""
        joint = f"{cls} {idv}".lower()
        if any(ind in joint for ind in BANNER_INDICATORS):
            return True
    # also check the element itself
    cls = " ".join(tag.get("class",[]) if isinstance(tag.get("class"), list) else ([tag.get("class")] if tag.get("class") else []))
    idv = tag.get("id","") or ""
    joint = f"{cls} {idv}".lower()
    if any(ind in joint for ind in BANNER_INDICATORS):
        return True
    return False

def scrub_text(s: str) -> str:
    if not s: return ""
    t = re.sub(r"\s+", " ", s).strip()
    return t

# -------------------------
# Core extraction: find actionable elements + context
# -------------------------
def extract_action_elements(soup: BeautifulSoup, base_url: str, company: str) -> List[Candidate]:
    """
    Find anchors, buttons, inputs of type button/submit, and elements with role=button.
    For each element, collect:
     - tag name, text, href (if any), outer HTML snippet
     - ancestor textual context (text of parent container up to N levels)
     - nearby headings within parent
     - attributes
    """
    candidates: List[Candidate] = []
    # build list of elements to consider
    tags = []
    tags.extend(soup.find_all("a"))
    tags.extend(soup.find_all("button"))
    tags.extend(soup.find_all("input", {"type": re.compile(r"^(button|submit)$", re.I)}))
    # elements with ARIA role=button
    tags.extend([t for t in soup.find_all(True) if t.has_attr("role") and t["role"].lower()=="button"])

    seen_elems = set()
    for el in tags:
        # dedupe by id + href + position
        key = (el.name, el.get("href"), el.get("id"), el.get("class"), scrub_text(el.get_text()))
        if key in seen_elems:
            continue
        seen_elems.add(key)

        # basic text
        text = scrub_text(el.get_text(" ", strip=True))
        href = el.get("href") if el.has_attr("href") else None
        outer = str(el)[:4000]  # cap size
        # ancestor container: climb up to 4 parents for context text
        ancestor = el
        context_text = ""
        nearby_headings = []
        for _ in range(4):
            if ancestor is None:
                break
            # collect headings inside ancestor
            for h in ancestor.find_all(["h1","h2","h3","h4"], limit=3):
                t = scrub_text(h.get_text())
                if t: nearby_headings.append(t)
            # collect text of ancestor for context
            ctxt = scrub_text(ancestor.get_text(" ", strip=True))
            if ctxt and len(ctxt) < 4000:
                context_text = ctxt
            ancestor = ancestor.parent

        # attributes dictionary
        attrs = {k: ( " ".join(v) if isinstance(v,list) else str(v) ) for k,v in el.attrs.items() }

        candidate = Candidate(
            company=company,
            base_url=base_url,
            element_tag=el.name,
            element_text=text,
            href=urljoin(base_url, href) if href else None,
            outer_html=outer,
            ancestor_text=context_text,
            nearby_headings=nearby_headings,
            attributes=attrs
        )

        candidates.append(candidate)

    return candidates

# -------------------------
# Heuristic filtering of candidates
# -------------------------
def candidate_is_potential_job(cand: Candidate, keywords:set=JOB_KEYWORDS, apply_terms:set=APPLY_TERMS) -> bool:
    """
    Return True if the candidate is plausibly a job posting action.
    We check:
     - The element text contains apply terms OR nearby context/heading contains job keywords
     - Exclude if inside a banner/nav/footer
     - Exclude if link looks like 'about/contact/shop'
     - Exclude single-word generic elements (unless apply term present)
    """
    # discard banner/nav-like
    try:
        if is_banner_element(BeautifulSoup(cand.outer_html, "html.parser")):
            return False
    except Exception:
        pass

    text = (cand.element_text or "").lower()
    context = (cand.ancestor_text or "").lower()
    headings = " ".join([h.lower() for h in cand.nearby_headings or []])
    href = (cand.href or "").lower()

    # skip non-http anchors that aren't useful
    if href and not href.startswith("http") and not href.startswith("/"):
        # e.g. mailto, javascript:void → ignore
        if ":" in href and not href.startswith("/"):
            return False

    # drop if link contains bad fragments
    if any(b in href for b in BAD_LINK_FRAGMENTS):
        return False

    # If element text contains apply terms, it's a high-probability candidate
    if any(term in text for term in apply_terms):
        # ensure not banner
        if len(text.split()) <= 1 and not any(k in context for k in keywords):
            # lone "apply" in nav could be ambiguous, prefer when context contains keywords
            return False
        return True

    # Else, check for keyword presence in element text, context, or headings
    keyword_found = any(k in text for k in keywords) or any(k in context for k in keywords) or any(k in headings for k in keywords)

    if not keyword_found:
        return False

    # Exclude titles that are too short / generic
    # Look for likely job title in headings/context (prefer headings)
    # For safety, require at least 2 words if the element text is generic
    candidate_title_source = text or headings or context
    if len(candidate_title_source.split()) < MIN_TITLE_WORDS:
        return False

    # finally, exclude typical banner words even if keywords present (e.g., product pages that mention a keyword)
    for bad in ("learn more","discover","shop","buy","pricing","features"):
        if bad in context or bad in text:
            return False

    # passed heuristics
    return True

# -------------------------
# Extract structured fields from candidate context
# -------------------------
def parse_structured_fields(cand: Candidate) -> Tuple[Optional[str], Optional[str], Optional[str], Optional[str]]:
    """
    Attempt to extract (title, location, salary, req_id, date_posted) from the candidate context.
    Returns mostly heuristics using regex on ancestor_text + nearby_headings.
    """
    ctx = " ".join(filter(None, [cand.element_text, " ".join(cand.nearby_headings or []), cand.ancestor_text or ""]))
    ctx = scrub_text(ctx)

    # tentative title: prefer nearby headings, then element_text, then first good chunk of ancestor
    title = None
    if cand.nearby_headings:
        title = cand.nearby_headings[0]
    if not title and cand.element_text and len(cand.element_text.split()) >= MIN_TITLE_WORDS:
        title = cand.element_text
    if not title:
        # try to guess a title-like line from context (longest short line)
        parts = [p.strip() for p in re.split(r'\n|\r|\||\u2022|-', ctx) if p.strip()]
        if parts:
            # pick the shortest non-generic part with at least 2 words
            for p in parts:
                if len(p.split()) >= MIN_TITLE_WORDS and not any(b in p.lower() for b in ("apply","learn more","about","contact")):
                    title = p
                    break

    # location
    loc = None
    loc_m = LOCATION_RE.search(ctx)
    if loc_m:
        loc = f"{loc_m.group(1)}, {loc_m.group(2)}"

    # salary
    sal = None
    sal_m = SALARY_RE.search(ctx)
    if sal_m:
        sal = sal_m.group(0)

    # requisition id
    req = None
    req_m = REQ_ID_RE.search(ctx)
    if req_m:
        req = req_m.group(2)

    # date posted
    date_posted = None
    date_m = DATE_RE.search(ctx)
    if date_m:
        date_posted = date_m.group(2).strip()

    return title, loc, sal, req, date_posted

# -------------------------
# High level pipeline
# -------------------------
def process_company_url(company: str, url: str,
                        keywords: Optional[List[str]] = None,
                        selenium_fallback: bool = True) -> List[JobMatch]:
    """
    For a single company URL:
     - Fetch page
     - Extract button/action elements and capture context
     - Filter candidates heuristically
     - Extract structured fields where possible
     - Return JobMatch list
    """
    keywords_set = set([k.lower() for k in (keywords or [])])
    # combine global JOB_KEYWORDS with user keywords so filter can use both
    all_keywords = JOB_KEYWORDS.union(keywords_set)

    html = fetch_html_requests(url)
    used_selenium = False
    if html is None and selenium_fallback:
        html = fetch_html_selenium(url)
        used_selenium = True
    if html is None:
        print(f"[{company}] Failed to fetch page.")
        return []

    soup = BeautifulSoup(html, "html.parser")
    candidates = extract_action_elements(soup, url, company)

    # Filter candidates
    potential = [c for c in candidates if candidate_is_potential_job(c, keywords=all_keywords, apply_terms=APPLY_TERMS)]

    # If none found and we haven't used selenium yet, try selenium-rendered page
    if not potential and selenium_fallback and not used_selenium:
        html2 = fetch_html_selenium(url)
        if html2:
            soup2 = BeautifulSoup(html2, "html.parser")
            candidates = extract_action_elements(soup2, url, company)
            potential = [c for c in candidates if candidate_is_potential_job(c, keywords=all_keywords, apply_terms=APPLY_TERMS)]

    # Build JobMatch objects with structured parsing
    matches: List[JobMatch] = []
    for cand in potential:
        title, loc, sal, req, date_posted = parse_structured_fields(cand)
        # fallback title: use element_text if parse_structured_fields couldn't find
        final_title = title or cand.element_text or (cand.nearby_headings[0] if cand.nearby_headings else None) or "Unknown"
        final_title = final_title.strip()
        link = cand.href or url
        jm = JobMatch(company=company, title=final_title, link=link, location=loc, salary=sal, req_id=req, date_posted=date_posted, context_html=cand.outer_html)
        matches.append(jm)

    # dedupe by link + title
    uniq = {}
    for m in matches:
        k = (m.link, m.title)
        if k not in uniq:
            uniq[k] = m
    return list(uniq.values())

# -------------------------
# Email composition & sending (kept simple)
# -------------------------
def compose_email_body(new_jobs_by_company: Dict[str, List[JobMatch]]) -> Tuple[str,str]:
    total = sum(len(v) for v in new_jobs_by_company.values())
    subject = f"[Jobs] {total} new matched job(s)"
    html_parts = [f"<p>Found <b>{total}</b> new matching job(s).</p>"]
    for comp, jobs in new_jobs_by_company.items():
        html_parts.append(f"<h3>{comp} ({len(jobs)})</h3><ul>")
        for j in jobs:
            meta = []
            if j.location: meta.append(j.location)
            if j.salary: meta.append(j.salary)
            if j.req_id: meta.append(f"Req: {j.req_id}")
            if j.date_posted: meta.append(j.date_posted)
            meta_str = f" ({', '.join(meta)})" if meta else ""
            html_parts.append(f'<li><a href="{j.link}">{j.title}</a>{meta_str}</li>')
        html_parts.append("</ul>")
    body = "\n".join(html_parts)
    return subject, body

def send_email_smtp(subject: str, body_html: str, smtp_config: dict):
    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = smtp_config["FROM"]
    msg["To"] = ", ".join(smtp_config["TO"])
    text_body = re.sub(r"<[^>]+>", "", body_html)
    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(body_html, "html"))

    sconf = smtp_config
    try:
        if sconf.get("PORT") == 465:
            server = smtplib.SMTP_SSL(sconf["HOST"], sconf["PORT"])
        else:
            server = smtplib.SMTP(sconf["HOST"], sconf.get("PORT",587))
            if sconf.get("USE_TLS", True):
                server.starttls()
        server.login(sconf["USERNAME"], sconf["PASSWORD"])
        server.sendmail(sconf["FROM"], sconf["TO"], msg.as_string())
        server.quit()
        print("[Email] Sent.")
    except Exception as e:
        print("[Email] Failed:", e)

# -------------------------
# Driver: run on CSV, filter by keywords, persist seen
# -------------------------
def run_from_csv(csv_path: str, keywords: List[str], smtp_config: Optional[dict]=None,
                 seen_path: str = DEFAULT_SEEN_PATH, selenium_fallback: bool = True):
    """
    csv file expected to have columns: company,url
    """
    import pandas as pd
    companies = pd.read_csv(csv_path)
    seen = load_seen(seen_path)
    new_seen = set(seen)
    all_new = {}

    for _, row in companies.iterrows():
        company = str(row["company"])
        url = str(row["url"])
        print(f"[Run] Scraping {company} -> {url}")
        matches = process_company_url(company, url, keywords=keywords, selenium_fallback=selenium_fallback)
        # filter by keywords strictly (title or context should contain keyword)
        filtered = []
        for m in matches:
            text_combo = " ".join(filter(None, [m.title, m.location or "", m.context_html or ""])).lower()
            if any(k.lower() in text_combo for k in keywords):
                key = (m.link, m.title)
                if key not in new_seen:
                    filtered.append(m)
                    new_seen.add(key)
        if filtered:
            all_new[company] = filtered
            print(f"  -> {len(filtered)} new matches for {company}")
        else:
            print("  -> No new matches")

    # persist seen
    save_seen(new_seen, seen_path)

    # send email if configured
    if smtp_config and all_new:
        subject, body = compose_email_body(all_new)
        send_email_smtp(subject, body, smtp_config)

    return all_new

# -------------------------
# Short discussion: ML approach
# -------------------------
ML_NOTE = """
Short note on ML: You can train a classifier (or fine-tune a transformer) to label DOM snippets as 'job-card' vs 'not-job'.
Pros:
 - Could generalize better than heuristics across many designs.
 - Can learn subtle cues (layout, phrases) humans miss.

Cons:
 - Requires labeled dataset: you'd need examples of many companies where DOM snippets are labeled job/not-job.
 - Needs feature extraction (DOM + text + visual hints), compute, monitoring and retraining.
 - For small-scale (30-200 companies) a hybrid heuristic + a few site-specific parsers + the 'apply-button' heuristic is far cheaper and fast to get running.

If you want to go ML later, a good path:
 - Harvest candidate snippets using this tool (both positives & negatives)
 - Label a few thousand examples (semi-automated)
 - Train a small classifier (e.g., text + DOM features or embeddings + small MLP)
 - Deploy to rank candidates before regex extraction.
"""

# -------------------------
# Example usage (commented)
# -------------------------
if __name__ == "__main__":
    # Example CSV path and email config (edit accordingly)
    CSV_PATH = "companies.csv"
    KEYWORDS = ["gnc","navigation","aerospace","satellite","propulsion","engineer","guidance"]
    SMTP = {
        "HOST": "smtp.example.com",
        "PORT": 587,
        "USERNAME": "you@example.com",
        "PASSWORD": "app-password-or-pw",
        "FROM": "you@example.com",
        "TO": ["you@example.com"],
        "USE_TLS": True
    }

    # run:
    results = run_from_csv(CSV_PATH, KEYWORDS, smtp_config=SMTP, selenium_fallback=True)
    print("Results summary:", {k: len(v) for k,v in results.items()})

SyntaxError: invalid syntax (3608177185.py, line 547)